In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

spark = SparkSession. \
    builder. \
    master('yarn'). \
    config('spark.ui.port', '0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    getOrCreate()

# Used Repatition and Partition By instead of (Optimize and ZORDER)

In [11]:
df_orders = spark.read \
.format("csv") \
.option('header',True) \
.load("/public/trendytech/orders_wh/orders_wh.csv")

In [12]:
df_orders.show(5)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
|       4|2013-07-25 00:00:...|       8827|         CLOSED|
|       5|2013-07-25 00:00:...|      11318|       COMPLETE|
+--------+--------------------+-----------+---------------+
only showing top 5 rows



In [17]:
df_orders.createOrReplaceTempView("orders_normal")

In [19]:
spark.sql("select * from orders_normal where order_status='CLOSED' ")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
4,2013-07-25 00:00:...,8827,CLOSED
12,2013-07-25 00:00:...,1837,CLOSED
18,2013-07-25 00:00:...,1205,CLOSED
24,2013-07-25 00:00:...,11441,CLOSED
25,2013-07-25 00:00:...,9503,CLOSED
37,2013-07-25 00:00:...,5863,CLOSED
51,2013-07-25 00:00:...,12271,CLOSED
57,2013-07-25 00:00:...,7073,CLOSED
61,2013-07-25 00:00:...,4791,CLOSED


In [14]:
df_orders.repartition(4).write \
.format("csv") \
.partitionBy("order_status") \
.save("/user/itv020178/optimized_files")

In [16]:
df_orders.createOrReplaceTempView("orders")

In [20]:
spark.sql("select * from orders_normal where order_status='CLOSED'")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
4,2013-07-25 00:00:...,8827,CLOSED
12,2013-07-25 00:00:...,1837,CLOSED
18,2013-07-25 00:00:...,1205,CLOSED
24,2013-07-25 00:00:...,11441,CLOSED
25,2013-07-25 00:00:...,9503,CLOSED
37,2013-07-25 00:00:...,5863,CLOSED
51,2013-07-25 00:00:...,12271,CLOSED
57,2013-07-25 00:00:...,7073,CLOSED
61,2013-07-25 00:00:...,4791,CLOSED
